In [26]:
import requests
import sys
import os

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

sys.path.insert(0, '..')  # Add parent directory for imports
from dotenv import load_dotenv
from model_config import ModelConfig

load_dotenv()

True

In [27]:
DOCS_BASE = "https://docs.langchain.com"

# Curated LangChain OSS pages for this tutorial. Expand this list or parse
# URLs from https://docs.langchain.com/llms.txt to index more of the site.
DOC_PATHS = [
    "oss/python/langchain/agents",
    "oss/python/deepagents/rag",
]

In [8]:
def load_langchain_docs(doc_paths: list[str] | None = None) -> list[Document]:
    """Fetch LangChain documentation pages as Documents."""
    paths = doc_paths or DOC_PATHS
    docs: list[Document] = []
    for path in paths:
        url = f"{DOCS_BASE}/{path}.md"
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
        except requests.RequestException:
            continue
        source = f"{DOCS_BASE}/{path}"
        docs.append(
            Document(page_content=response.text, metadata={"source": source})
        )
    return docs


docs = load_langchain_docs()
print(f"Loaded {len(docs)} documentation pages.")

Loaded 2 documentation pages.


In [14]:
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"Total characters: {total_chars}")
print(docs[0].page_content[:400])

Total characters: 127443
> ## Documentation Index
> Fetch the complete documentation index at: https://docs.langchain.com/llms.txt
> Use this file to discover all available pages before exploring further.

# Agents

An agent is a model calling tools in a loop until a given task is complete.

<img src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/core_agent_loop.svg?fit=max&auto=format&n=jtty0O--UJOKG


In [28]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
print(f"Split documentation into {len(all_splits)} chunks.")

Split documentation into 166 chunks.


In [29]:
#embeddings = ModelConfig.openrouter("liquid/lfm-2.5-embedding-350m:free")

In [42]:
embeddings = OpenAIEmbeddings(
            model='liquid/lfm-2.5-embedding-350m:free',
            api_key=os.getenv("ROUTER_KEY"),
            base_url="https://openrouter.ai/api/v1",
        )

In [47]:
type(all_splits)

list

In [43]:
cwd=os.getcwd()
db_dir=os.path.join(cwd,'db','chroma')

In [50]:
vector_store = Chroma(collection_name = "langchain_docs", 
                      embedding_function = embeddings, 
                      persist_directory = db_dir)

In [51]:
vector_store.add_documents(all_splits)

BadRequestError: Error code: 400 - {'error': {'message': 'HTTP 400: {"error":{"message":"Invalid input","type":"invalid_request_error","param":"input","code":null}}', 'code': 400}}